# 03 — Stage 7 â€” Empirical range and response experiment

Use a **CPU, High-RAM** Colab runtime. This reuses the selected LightGBM family for new explicit targets; it does not rerun model-family search or solver work. Reveal-selected range assumptions and observed response predictions remain diagnostics with zero strategy authority. Whole-game membership and source hashes are frozen in this package. No PokerNow actions. Run top-to-bottom.


In [ ]:
import os, sys, subprocess, pathlib
assert os.path.isdir("/content") and os.environ.get("COLAB_RELEASE_TAG"), "Colab runtime required"
assert sys.version_info[:2] == (3, 12), "Select the frozen Python 3.12 runtime before mounting Drive"
meminfo = dict(line.split(":",1) for line in pathlib.Path("/proc/meminfo").read_text().splitlines())
actual_ram_bytes = int(meminfo["MemTotal"].split()[0]) * 1024
assert actual_ram_bytes >= 48 * 1000**3, "Select CPU High-RAM (at least 48 GB) before mounting Drive"
assert os.environ.get("COLAB_GPU", "0") in ("", "0"), "Select the frozen CPU runtime"
print({"runtime_python":sys.version,"actual_ram_bytes":actual_ram_bytes,"actual_ram_gb":actual_ram_bytes/1e9,"actual_ram_gib":actual_ram_bytes/1024**3,"ram_role":"frozen runtime shape prerequisite; not a worker cap"})
from google.colab import drive
drive.mount("/content/drive")
subprocess.check_call([sys.executable,"-m","pip","install","--quiet","numpy==2.2.6","scikit-learn==1.7.2","lightgbm==4.6.0","threadpoolctl==3.6.0"])


In [ ]:
import hashlib, json, pathlib, zipfile, shutil, uuid
STAGE = pathlib.Path('/content/drive/MyDrive/Poker Assistant Model/Stage 7 - Empirical Assistant Layers')
PACKAGE = STAGE / "inputs" / 'stage6_empirical_range_learning_v2_stage7_range_response_v2_r2.zip'
ROOT = pathlib.Path("/content") / 'range90_stage7_range_response_v2_r2'
ROOT.mkdir(exist_ok=True)
def digest(path):
    h=hashlib.sha256()
    with path.open("rb") as stream:
        for part in iter(lambda:stream.read(1048576),b""): h.update(part)
    return h.hexdigest()
assert PACKAGE.stat().st_size == 93349 and digest(PACKAGE) == '0d607e5c73b5756caef18e1ffb54a15119f2a9ef7904f158a2ca60396b7b8254'
MEMBERS = json.loads('{"release_plan.json":{"bytes":124541,"sha256":"63b2fef4556ca453072f46a81cf0ba5fa39a2e48bc3fbf31d22c12d5fc663a9b"},"stage6_empirical_range_learning_v2.py":{"bytes":57344,"sha256":"7d18c9249f8f57c070c6208c792568144e6052c985bd87a4947198d46c085aac"},"stage6_empirical_range_learning_v2_adapter.py":{"bytes":27538,"sha256":"427c8b5aa50f21a7a1af86ec7855e704dac87a80789532f167d57c1aaccb9b55"},"stage6_empirical_range_learning_v2_release.py":{"bytes":47804,"sha256":"400ac7f19e3a45b2f5959230ac393958863fc88270679de6473356de795a2c3d"},"stage6_empirical_range_learning_v2_streaming.py":{"bytes":21067,"sha256":"6d42e2f920f3f2f5ec12ce963d5d4fba694db6aa1e0e89b69fb64f99b7341544"}}')
with zipfile.ZipFile(PACKAGE) as archive:
    assert len(archive.namelist())==len(set(archive.namelist())) and set(archive.namelist())==set(MEMBERS)
    for name, expected in MEMBERS.items():
        assert pathlib.PurePosixPath(name).name==name
        raw=archive.read(name)
        assert len(raw)==expected["bytes"] and hashlib.sha256(raw).hexdigest()==expected["sha256"]
        target=ROOT/name
        if target.exists(): assert target.read_bytes()==raw
        else:
            partial=target.with_name(target.name+".partial."+uuid.uuid4().hex)
            with partial.open("xb") as stream:
                stream.write(raw); stream.flush(); os.fsync(stream.fileno())
            assert partial.stat().st_size==expected["bytes"] and digest(partial)==expected["sha256"]
            try: os.link(partial,target)
            except FileExistsError: assert target.read_bytes()==raw
            partial.unlink()
print("Authenticated release and source code", ROOT)

def adopt_prepared(predecessor, root, expected_plan, expected_receipt):
    """Copy only frozen source/preparation; fitting starts in the new namespace."""
    import hashlib
    import json
    import os
    from pathlib import Path
    import shutil
    import uuid

    def identity(path):
        path = Path(path); digest = hashlib.sha256()
        with path.open("rb") as stream:
            for part in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(part)
        return dict(bytes=path.stat().st_size, sha256=digest.hexdigest())

    def verify(path, expected):
        if identity(path) != expected:
            raise ValueError("prepared_reuse_identity_mismatch:" + str(path))

    predecessor, root = Path(predecessor), Path(root)
    if predecessor.resolve() == root.resolve():
        raise ValueError("prepared_reuse_requires_new_namespace")
    verify(predecessor / "release_plan.json", expected_plan)
    old = json.loads((predecessor / "release_plan.json").read_text())
    current = json.loads((root / "release_plan.json").read_text())
    adapter_name = "stage6_empirical_range_learning_v2_adapter.py"
    if old["inputs"] != current["inputs"] or old["code_files"][adapter_name] != current["code_files"][adapter_name]:
        raise ValueError("prepared_reuse_source_or_adapter_changed")
    verify(predecessor / adapter_name, current["code_files"][adapter_name])
    verify(root / adapter_name, current["code_files"][adapter_name])
    previous = predecessor / "work"
    receipt_path = previous / "prepared_rows/range_adapter_receipt.json"
    verify(receipt_path, expected_receipt)
    receipt = json.loads(receipt_path.read_text())
    source_inputs = {}
    for role, item in current["inputs"].items():
        source_inputs[role] = (dict(bytes=item["member_bytes"], sha256=item["member_sha256"])
                               if item.get("member") is not None
                               else {key: item[key] for key in ("bytes", "sha256")})
    lineage = receipt["lineage"]
    if (lineage["source_inputs"] != source_inputs or lineage["synthetic_fixture"] is not False
            or lineage["adapter_sha256"] != current["code_files"][adapter_name]["sha256"]):
        raise ValueError("prepared_reuse_lineage_mismatch")
    if set(receipt["members"]) != {"range_training_rows.jsonl", "range_quarantine_rows.jsonl", "identity_mapping.json"}:
        raise ValueError("prepared_reuse_inventory_mismatch")
    manifest = previous / "input_manifest.json"
    if (identity(manifest)["sha256"] != lineage["input_manifest_sha256"]
            or json.loads(manifest.read_text()) != source_inputs):
        raise ValueError("prepared_reuse_input_manifest_mismatch")
    members = {"source/" + role + (".jsonl" if role == "hands" else ".csv"): item
               for role, item in source_inputs.items()}
    members.update({"prepared_rows/" + name: item for name, item in receipt["members"].items()})
    members["input_manifest.json"] = identity(manifest)
    members["prepared_rows/range_adapter_receipt.json"] = expected_receipt
    for name, expected in members.items():
        verify(previous / name, expected)
    for name, expected in members.items():
        source, destination = previous / name, root / "work" / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            partial = destination.with_name(destination.name + ".partial." + uuid.uuid4().hex)
            with source.open("rb") as inp, partial.open("xb") as out:
                shutil.copyfileobj(inp, out, 1024 * 1024)
                out.flush(); os.fsync(out.fileno())
            verify(partial, expected)
            try:
                os.link(partial, destination)
            except FileExistsError:
                verify(destination, expected)
            partial.unlink()
        verify(destination, expected); verify(source, expected)
        print(json.dumps(dict(phase="reuse_authenticated_preparation", member=name, **expected)), flush=True)
    return dict(predecessor_plan=expected_plan, adapter_receipt=expected_receipt,
                members=members, source_preparation_recomputed=False, fitting_reused=False)

REUSED=adopt_prepared(pathlib.Path('/content/range90_stage7_range_response_v2_r1'), ROOT, {'bytes': 124541, 'sha256': '7cdcbf1b87ee2cd631f679ac2f2a9a488d5cbf17a3aa4ad91faaf225b438d55b'}, {'bytes': 92313, 'sha256': 'e23b29efdbc2f1e399fb1361e480734f756a8e80f350bd4f2dbb837c123c825b'})
print({'phase':'completed_source_preparation_reused','source_preparation_recomputed':False,'fitting_reused':False})


The r2 release preserves the completed r1 source and prepared rows by exact input, adapter, receipt and member hashes. Dealt-player identity is separated from shared profile ownership. Only compact preparation and fitting restart in this new namespace. The original prepared rows, labels, cohorts and model settings remain unchanged.


In [ ]:
process=subprocess.Popen([sys.executable,str(ROOT/"stage6_empirical_range_learning_v2_release.py"),"run-colab","--root",str(ROOT)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout: print(line,end="",flush=True)
returncode=process.wait()
if returncode: raise subprocess.CalledProcessError(returncode,process.args)


The direct result ZIP and publication receipt are in Stage 7 `returned_results`. Two raw-byte Drive readbacks are required by the producer. The runner retrieves the exact Drive object and the implementation task performs strict import and independent review. A completed notebook does not promote any range, strategy or model.
